# 02 — Reference scVI and scANVI models

**What this does:** trains the unsupervised donor-conditioned scVI model on
the 13 reference donors, then initialises semi-supervised scANVI from it.

**Architecture** (current scvi-tools reference-mapping tutorial): layer
norm, no batch norm, encoded covariates, 2 layers, dropout 0.2, 30 latent
dimensions, ZINB.

**Leakage controls:** only reference cells are loaded; the query donor (D6)
is absent; no query labels exist anywhere in this notebook.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
import heartmap
print("heartmap from:", Path(heartmap.__file__).parent)


In [ ]:
import anndata as ad
from heartmap.config import load_config
from heartmap.data import validate_counts
cfg = load_config("configs/main.yaml")
reference = ad.read_h5ad(cfg.reference_path)
validate_counts(reference, cfg["counts_layer"])
print(reference.shape, "donors:",
      sorted(reference.obs[cfg["donor_key"]].astype(str).unique()))
assert "D6" not in set(reference.obs[cfg["donor_key"]].astype(str))


## 1. Subset to reference-only HVGs

In [ ]:
from heartmap.models import subset_hvg
ref_h = subset_hvg(reference, cfg)
print("reference on", ref_h.n_vars, "reference-selected HVGs")


## 2. Train scVI (up to 400 epochs, early stopping patience 20)

Raw counts enter through `layer='counts'`; donor is the batch covariate. On CPU this takes tens of minutes; on a Colab T4 a few minutes.

In [ ]:
from heartmap.models import train_scvi_reference, save_history
(cfg.results_dir / "training").mkdir(parents=True, exist_ok=True)
scvi_model, scvi_history = train_scvi_reference(ref_h, cfg)
save_history(scvi_model, cfg.results_dir / "training" /
             f"scvi_history_{cfg.run_tag}.csv")


## 3. Training curve

Healthy behaviour: loss drops steeply then plateaus; early stopping ends training when validation loss stops improving.

In [ ]:
import pandas as pd
hist = pd.read_csv(cfg.results_dir / "training" /
                   f"scvi_history_{cfg.run_tag}.csv")
ax = hist[["train_loss_epoch", "validation_loss"]].dropna(how="all").plot(
    figsize=(7, 3))
ax.set_xlabel("epoch"); ax.set_ylabel("ELBO loss")


## 4. Train scANVI from the scVI weights (max 20 epochs)

`labels_key='labels_scanvi'` contains the 11 reference broad types; `unlabeled_category='Unknown'`.

In [ ]:
from heartmap.models import train_scanvi_reference
scanvi_model, scanvi_history = train_scanvi_reference(
    scvi_model, ref_h, cfg)
save_history(scanvi_model, cfg.results_dir / "training" /
             f"scanvi_history_{cfg.run_tag}.csv")


## 5. Persist models and reference latent

The latent AnnData carries donor and true reference labels (reference labels are allowed here) and is used in notebook 04 for the joint UMAP.

In [ ]:
import anndata as ad
from heartmap import LABELS_KEY
scvi_dir = cfg.models_dir / f"scvi_reference_{cfg.run_tag}"
scanvi_dir = cfg.models_dir / f"scanvi_reference_{cfg.run_tag}"
cfg.models_dir.mkdir(exist_ok=True)
scvi_model.save(str(scvi_dir), overwrite=True)
scanvi_model.save(str(scanvi_dir), overwrite=True)

ref_h.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
ref_latent = ad.AnnData(
    X=ref_h.obsm["X_scANVI"].copy(),
    obs=ref_h.obs[[cfg["donor_key"], cfg["cell_type_key"],
                   LABELS_KEY]].copy())
ref_latent.obs["split"] = "reference"
ref_latent.write_h5ad(
    cfg.results_dir / f"reference_scanvi_latent_{cfg.run_tag}.h5ad")
print("saved models and reference latent")


Training is complete. The sealed query donor has never been touched. Continue to notebook 03 for the scArches update.